In this notebook we assemble gene cohorts from condition cohorts

In [1]:
import pandas as pd
from pathlib import Path

condition_cohorts_dir = 'condition_cohorts'
gene_cohorts_dir = 'gene_cohorts'

In [2]:
# Mapping of genes to phenotypes:
# gene: (case phenotypes, control phenotypes, inheritence)
gene_phenotypes = {
    'BAP1': (
        {
            'melanoma',
            'mesothelioma',
            'renal cancer',
            'basal cell carcinoma',
            'intracranial meningioma',
            'cholangiocarcinoma of the biliary tract'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'BARD1': (
        {'breast cancer', 'ovarian cancer'},
        {'cancer'},
        'autosomal dominant'
    ),
    'BRCA1': (
        {
            'breast cancer',
            'ovarian cancer',
            'prostate cancer',
            'pancreatic cancer',
            'melanoma'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'BRCA2': (
        {
            'breast cancer',
            'ovarian cancer',
            'prostate cancer',
            'pancreatic cancer',
            'melanoma'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'CALM1': (
        {'long QT syndrome'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'CALM2': (
        {'long QT syndrome'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'CALM3': (
        {'long QT syndrome'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    #'CARD11',
    'CBS': ({'homocystinuria'}, {'homocystinuria'}, 'autosomal recessive'),
    'CHEK2': (
        {
            'breast cancer',
            'prostate cancer'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    #'CRX',
    'CTCF': ({'intellectual disability'}, {'intellectual disability'}, 'autosomal dominant'),
    'DDX3X': ({'intellectual disability male'}, {'intellectual disability male'}, 'X-linked'),
    #'F9': ({'hemophilia B'}, {'hemophilia'}, 'X-linked'),
    #'FKRP',
    'G6PD': (
        {'deficiency of glucose-6-phosphate dehydrogenase'},
        {
            'deficiency of glucose-6-phosphate dehydrogenase',
            'thalassemia',
            'sickle cell hemoglobin disease'
        },
        'X-linked' # Abbye: X-linked dominant
    ),
    'GCK': (
        {'MODY'},
        {'diabetes'},
        'autosomal dominant'
    ),
    #'HMBS',
    #'JAG1',
    'KCNE1': (
        {'long QT syndrome', 'sudden cardiac death'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'KCNH2': (
        {'long QT syndrome', 'sudden cardiac death'},
        {'abnormal QT interval', 'cardiac arrhythmia', 'sudden cardiac death'},
        'autosomal dominant'
    ),
    'KCNQ4': (
        {'nonsyndromic genetic hearing loss'},
        {'hearing loss'},
        'autosomal dominant'
    ),
    #'LARGE1',
    'MSH2': (
        {
            'colorectal cancer',
            'ovarian cancer',
            'stomach cancer',
            'small bowel cancer',
            'urinary tract cancer',
            'prostate cancer',
            'brain cancer',
            'endometrial cancer'
        },
        {'cancer'},
        'autosomal dominant' # AR for mismatch repair- maybe we should also consider AR analysis
    ),
    #'NDUFAF6',
    'OTC': (
        {'ornithine carbamoyltransferase deficiency'},
        {'disorder of the urea cycle metabolism', 'hyperammonemia'},
        'X-linked' # Males severely affected from birth, females asymptomatic to severe
    ),
    'PALB2': (
        {
            'breast cancer',
            'ovarian cancer',
            'pancreatic cancer',
            'prostate cancer'
        },
        {'cancer'},
        'autosomal dominant'
    ),
    #'PAX6',
    'PTEN': (
        {
            'breast cancer',
            'endometrial cancer',
            'thyroid cancer',
            'colorectal cancer',
            'renal cancer',
            'melanoma',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'RAD51C': (
        {
            'breast cancer',
            'ovarian cancer',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    'RAD51D': (
        {
            'breast cancer',
            'ovarian cancer',
        },
        {'cancer'},
        'autosomal dominant'
    ),
    #'RHO',
    'SCN5A':(
        {
            'long QT syndrome',
            'dilated cardiomyopathy',
            'sudden cardiac death'
        },
        {
            'abnormal QT interval',
            'cardiomyopathy',
            'cardiac arrhythmia',
            'sudden cardiac death'
        },
        'autosomal dominant'
    ),
    'SFPQ': ({'renal cell carcinoma or bone sarcoma'}, {'cancer'}, 'autosomal dominant'),
    'SGCB': ({'limb-girdle muscular dystrophy'}, {'muscular dystrophy'}, 'autosomal recessive'),
    'TARDBP': (
        {
            'amyotrophic lateral sclerosis',
            'frontotemporal dementia'
        },
        {
            'amyotrophic lateral sclerosis',
            'frontotemporal dementia'
        },
        'autosomal dominant'
    ),
    'TP53': (
        {'cancer'},
        {'cancer'},
        'autosomal dominant'
    ),
    #'TPK1',
    'TSC2': (
        {'tuberous sclerosis'},
        {'tuberous sclerosis', 'cancer'},
        'autosomal dominant'
    ),
    #'VHL',
    'XRCC2': (
        {'breast cancer', 'ovarian cancer'},
        {'cancer'},
        'autosomal dominant'
    )
}


# List of genes included in the Pillar Project Publication (P3)
p3_genes = [
    'BAP1',
    'BARD1',
    'BRCA1',
    'BRCA2',
    'CHEK2',
    'GCK',
    'KCNE1',
    'KCNH2',
    'KCNQ4',
    'MSH2',
    'PALB2',
    'PTEN',
    'RAD51C',
    'RAD51D',
    'SCN5A',
    'TP53',
    'TSC2'
]

In [3]:
for gene in p3_genes:

    print(f'Assembing cohort for {gene}...')

    out_path = Path(gene_cohorts_dir, f'{gene} cohort.parquet')
    if out_path.exists():
        print(f'{out_path} exists, skipping.')
        continue

    case_phenos, control_phenos, _ = gene_phenotypes[gene]

    try:
        case_dfs = [
            pd.read_table(
                f'{condition_cohorts_dir}/{case_cohort} case pool.tsv.gz',
                usecols=['person_id'],
                index_col='person_id'
            ).assign(case=1)
            for case_cohort in case_phenos
        ]
    
        controls = None
        for control_cohort in control_phenos:
            df = pd.read_table(f'{condition_cohorts_dir}/{control_cohort} control pool.tsv.gz',
                usecols=['person_id'],
                index_col='person_id'
            )
            if controls is None:
                controls = df
            else:
                controls = controls.join(df, how='inner')
    except FileNotFoundError as e:
        print(e)
        continue
    
    gene_cohort = pd.concat(
        case_dfs + [controls.assign(case=0)]
    )

    gene_cohort.to_parquet(out_path)
    print(f'Saved to {out_path}')